# 12_01 The next token: what does a language model actually compute?

A chat model seems to write answers. What it computes is much smaller: one score for every token in its
vocabulary, for the single position after the text so far. In this notebook you look at those scores,
turn them into probabilities, write the loop that turns one prediction into a sentence, and see what
temperature and top-p do to the choice.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Loading the model takes about half a minute the first time in a session, less once the session's warm-up has read it from disk.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-12-large-and-small-language-models", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'bm25s': 'bm25s',
           'sentence_transformers': 'sentence-transformers',
           'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json, time
import torch
import slm
from nlpcheck import ask, guess, reveal, check_12_01

t = time.time()
tok, model = slm.load()          # SmolLM2-360M-Instruct, from the image; nothing downloads
print(f"model loaded in {time.time() - t:.0f} s: {sum(p.numel() for p in model.parameters()):,} weights, "
      f"a vocabulary of {len(tok):,} tokens")

## 1. Recall

**r1.** In Lab 11, what was BERT trained to do? (a) predict the next word from the words before it,
(b) fill in masked words using the words on both sides, (c) translate between languages

**r2.** What does softmax do to a list of scores? (a) makes them positive and add up to 1,
(b) keeps only the largest, (c) sorts them

In [ ]:
ask("r1", "")
ask("r2", "")

## 2. Tokens, not words

The model never sees letters or words. It sees token ids from a vocabulary of 49,152 pieces learned
from its training text (byte-level BPE, as in Lab 01). A piece often carries its leading space.

In [ ]:
for text in ["My phone won't", "Kittiwake Mobile"]:
    ids = tok(text, add_special_tokens=False).input_ids
    print(f"{text!r:22} -> {ids}  {[tok.decode([i]) for i in ids]}")

Common words are one token each; the invented name "Kittiwake" is four pieces, because it never
appeared often enough in the training text to earn a token of its own.

## 3. One forward pass, one set of scores

A Kittiwake support chat opens with "My phone won't". What single token does the model think comes next,
and with what probability? Write your guess as the word, for example `"work"`.

In [ ]:
guess("next_token", None)   # the most likely next word, as a string

In [ ]:
text = "My phone won't"
ids = tok(text, add_special_tokens=False).input_ids
with torch.no_grad():
    logits = model(torch.tensor([ids])).logits[0, -1]   # the scores for the position after the last token
print("logits:", tuple(logits.shape), "one per vocabulary entry")

e = (logits - logits.max()).exp()                          # softmax, written out; subtracting the largest
probs = e / e.sum()                                        # logit first changes nothing but avoids overflow
print("largest difference from torch.softmax:", float((probs - logits.softmax(-1)).abs().max()))
top = probs.topk(5)
for p, i in zip(top.values, top.indices):
    print(f"  {tok.decode([int(i)])!r:12} logit {logits[i]:6.2f}   probability {p:.3f}")
reveal("next_token", tok.decode([int(top.indices[0])]).strip())

`' turn'`, with a probability of only about 0.24, and `' work'` close behind at 0.17. The model does not
choose a word: it spreads its belief across the whole vocabulary, and here nearly half of it lands outside
the top five. A **logit** is the raw score; softmax exponentiates each one and divides by the total, so a
logit one point higher means a probability about 2.7 times larger (`' turn'` at 22.2 against `' work'` at
21.8 is a factor of only 1.4).

`model(...)` also returned scores for every earlier position: what it would have predicted after "My",
after "My phone", and so on. Training uses all of them at once; generating uses only the last.

## 4. From one prediction to a sentence

Generating text is that one step in a loop: pick a token, append it, run the model again on the longer
sequence. Picking the most likely token every time is **greedy decoding**. The worked example takes one
step:

In [ ]:
with torch.no_grad():
    logits = model(torch.tensor([ids])).logits[0, -1]
next_id = int(logits.argmax())
print("one greedy step:", repr(text + tok.decode([next_id])))

Your turn: eight steps. Inside the loop, run the model on everything so far (`ids + greedy_ids`), take the
argmax of the last position's logits, and let the loop append it. It runs the whole sequence again each
step, which is slower than `generate()` (that keeps a cache of earlier positions) but gives the same
tokens. It takes a few seconds.

In [ ]:
greedy_ids = []
for step in range(8):
    next_id = None   # YOUR CODE HERE: logits for ids + greedy_ids, then the argmax of the last position
    if next_id is None:
        break
    greedy_ids.append(next_id)
print("by hand:   ", repr(text + tok.decode(greedy_ids)))

with torch.no_grad():
    out = model.generate(torch.tensor([ids]), max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
generate_ids = out[0, len(ids):].tolist()
print("generate():", repr(text + tok.decode(generate_ids)))

## 5. Temperature: the same prompt, different answers

Chat products do not always take the top token; they **sample** from the probabilities. The next cell asks
for a slogan three times at temperature 0 and three times at temperature 1.5. How many *different* slogans
will each give? Guess two numbers, for example `(3, 3)`. It takes about a minute.

In [ ]:
guess("distinct_slogans", None)   # (distinct at T = 0, distinct at T = 1.5)

In [ ]:
ask_slogan = [{"role": "user", "content": "Write a one-line slogan for a small mobile network."}]
runs = {}
for T in (0.0, 1.5):
    runs[T] = [slm.generate(ask_slogan, max_new_tokens=20, temperature=T, seed=s) for s in range(3)]
    for r in runs[T]:
        print(f"T = {T}: {r}")
reveal("distinct_slogans", (len(set(runs[0.0])), len(set(runs[1.5]))))

`(1, 3)`: at temperature 0 the three runs are the same slogan, because greedy decoding has no randomness
in it. At 1.5 they are three different strings, and none of them is a slogan: fragments of words and
symbols. Sampling at a high temperature did not make the model more creative, it made it pick tokens it
thought were unlikely, and after a few of those the text has left anything it has seen.

**Temperature** divides every logit by `T` before the softmax. Below 1 the gaps grow and the top token
takes more of the probability; above 1 the gaps shrink and the tail gains. **Top-p** (nucleus sampling)
then keeps only the smallest set of most likely tokens whose probabilities add up to `p`, and samples
from those. Your turn: write both, on the "My phone won't" logits.

In [ ]:
logits = slm.last_logits("My phone won't")

def with_temperature(logits, T):
    return None   # YOUR CODE HERE: divide the logits by T, then softmax

def top_p_count(probs, p):
    s = probs.sort(descending=True).values
    c = s.cumsum(0)
    return None   # YOUR CODE HERE: how many tokens until c reaches p, counting the one that crosses it

top1, kept = {}, {}
for T in (0.5, 1.0, 1.5):
    pr = with_temperature(logits, T)
    if pr is not None:
        top1[str(T)] = round(float(pr.max()), 4)
        n = top_p_count(pr, 0.9)
        if n is not None and T != 0.5:
            kept[str(T)] = int(n)
print("top token's probability:", top1)
print("tokens top-p 0.9 keeps:  ", kept)

In [ ]:
from nlpcheck import ref_temperature, ref_top_p
slm.save_json("12_01_next_token.json", {
    "greedy_ids": greedy_ids, "generate_ids": generate_ids,
    "top1_at_T": top1, "top_p_kept": kept,
    "reference_top1_at_T": {str(T): round(float(ref_temperature(logits, T).max()), 4) for T in (0.5, 1.0, 1.5)},
    "reference_top_p_kept": {str(T): ref_top_p(ref_temperature(logits, T), 0.9) for T in (1.0, 1.5)},
    "slogans": {str(k): v for k, v in runs.items()}})
check_12_01()

At temperature 1, top-p 0.9 samples from about 33 tokens; at 1.5 from about 550, which is why the slogans
fell apart. Production settings sit around temperature 0.7 with top-p 0.9 for writing, and 0 for anything
a program will parse or a person will act on, such as an answer about a price.

## 6. Exit ticket

**x1.** What does top-p 0.9 do? (a) keeps the top 90 percent of tokens by count, (b) multiplies every
probability by 0.9, (c) samples only from the fewest most likely tokens whose probabilities add up to 0.9

In [ ]:
ask("x1", "")

Explain it back: the model gave `' turn'` a probability of 0.24. Why is "the model thinks the next word is
turn" a misleading way to describe that? One or two sentences.

*Your explanation:* 